# Ian's Project

## Data Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget -q --show-progress "https://raw.githubusercontent.com/IanRmacdonnell/Short-story-storage/main/preprocessed_data.csv"
!wget -q --show-progress "https://raw.githubusercontent.com/IanRmacdonnell/Short-story-storage/main/Short_Stories_with_Image_Descriptions.csv"
#!pip install openai==0.28
!pip install openai
!pip install spacy
!python -m spacy download en_core_web_sm

#create a chatgpt key- use files- api key- open ai google collab

In [ ]:
# Imports
import numpy as np
import pandas as pd
import spacy

## Data Preprocessing

In [ ]:
data = pd.read_csv("Short_Stories_with_Image_Descriptions.csv")
data.head()

In [ ]:
data.shape

In [ ]:
Classifications=list(pd.unique(data["classification"]))
print(Classifications)
len("Classifications")

In [ ]:
import numpy as np
import pandas as pd

# Load the data
data = pd.read_csv("https://raw.githubusercontent.com/IanRmacdonnell/Short-story-storage/main/preprocessed_data.csv")

# Display the first few rows of the dataset
print(data.head())

# Display the shape of the dataset
print("Shape of the dataset:", data.shape)

# Get unique classifications
Classifications = list(pd.unique(data["classification"]))
print("Classifications:", Classifications)

In [ ]:
# Basic statistics of the dataset
print(data.describe())

# Information about the dataset
print(data.info())

# Count of each classification
print(data['classification'].value_counts())

In [ ]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation

# Download stopwords if not already downloaded
import nltk
nltk.download('stopwords')
nltk.download('punkt')

# Preprocessing function
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(f"[{re.escape(punctuation)}]", "", text)
    # Tokenize text
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

# Apply preprocessing to the 'text' column
data['processed_text'] = data['text'].apply(preprocess_text)

# Display the first few rows of the processed data
print(data.head())

In [ ]:
from textblob import TextBlob

# Function to get the sentiment of a text
def get_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

# Apply sentiment analysis to the 'text' column
data['polarity'], data['subjectivity'] = zip(*data['text'].apply(get_sentiment))

# Display the first few rows with sentiment analysis
print(data.head())

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Function to get the word frequency for a given classification
def get_word_frequency(classification):
    text = " ".join(data[data['classification'] == classification]['processed_text'].apply(lambda x: " ".join(x)))
    word_freq = Counter(text.split())
    return word_freq

# Get word frequency for each classification
for classification in Classifications:
    word_freq = get_word_frequency(classification)
    print(f"Word frequency for {classification}:")
    print(word_freq.most_common(10))

    # Plot the word frequency
    common_words = word_freq.most_common(10)
    words, counts = zip(*common_words)
    plt.figure(figsize=(10, 5))
    plt.bar(words, counts)
    plt.title(f"Top 10 Words in {classification}")
    plt.show()

##Genre Classification


In [ ]:
import numpy as np
import pandas as pd
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import string
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

!python -m spacy download en_core_web_md
import en_core_web_md
text_to_nlp = spacy.load('en_core_web_md')

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

STOP_WORDS = stopwords.words('english')

# quiet future deprecation warnings
import warnings
warnings.filterwarnings('ignore')

#function that we brought from NLP practice notebook
def tokenize(text):
    clean_tokens = []
    for token in text_to_nlp(text):
        if (not token.is_stop) & (token.lemma_ != '-PRON-') & (not token.is_punct): # -PRON- is a special all inclusive "lemma" spaCy uses for any pronoun, we want to exclude these
            clean_tokens.append(token.lemma_)
    return clean_tokens
X = data['text']
bow_transformer = CountVectorizer(analyzer=tokenize, max_features=800).fit(X)
X = bow_transformer.transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## trying out NLP's


XG BOOST MODEL

In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
import nltk

#stopwords
nltk.download('stopwords')
nltk.download('punkt')

# Load the data
data = pd.read_csv("https://raw.githubusercontent.com/IanRmacdonnell/Short-story-storage/main/preprocessed_data.csv")

# Display the first few rows
print(data.head())

# Display the shape
print("Shape of the dataset:", data.shape)

# classifications
Classifications = list(pd.unique(data["classification"]))
print("Classifications:", Classifications)

# Basic stats
print(data.describe())

# Information
print(data.info())

# Count of each classification
print(data['classification'].value_counts())

# Preprocessing function
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(f"[{re.escape(punctuation)}]", "", text)
    # Tokenize text
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing to the 'text' column
data['processed_text'] = data['text'].apply(preprocess_text)

# Display the first few rows
print(data.head())

# Transform the text data using TF-IDF vectorizer - https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(data['processed_text'])

# Get the target label
y = data['classification']

# Split the data into training and testing sets - copied from notebook
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the MLPClassifier - https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)

# Train the MLPClassifier - https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier
mlp.fit(X_train, y_train)

# Make predictions on the test data
y_pred_mlp = mlp.predict(X_test)

# Evaluate the MLP model's performance
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
print("Multi-Layer Perceptron Classifier Accuracy:", accuracy_mlp)
print("Multi-Layer Perceptron Classifier Report:\n", classification_report(y_test, y_pred_mlp))


Random Forest classifier

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
import matplotlib.pyplot as plt
from gensim.models import Word2Vec

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')

# Load and preprocess data
data = pd.read_csv("https://raw.githubusercontent.com/IanRmacdonnell/Short-story-storage/main/preprocessed_data.csv")
data['processed_text'] = data['text'].apply(lambda x: " ".join(
    [word for word in word_tokenize(re.sub(f"[{re.escape(punctuation)}]", "", x.lower())) if word not in set(stopwords.words('english'))]
))

# Train Skip-gram model
sentences = data['processed_text'].apply(lambda x: x.split()).tolist()
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, sg=1, min_count=1, workers=4)
word2vec_model.save("skipgram.model")

# Transform text data using Skip-gram model
def get_average_word_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

data['average_word_vector'] = data['processed_text'].apply(lambda x: get_average_word_vector(x.split(), word2vec_model))
X = np.array(data['average_word_vector'].tolist())

# Encode labels and handle rare classes
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(data['classification'])
class_counts = pd.Series(y_encoded).value_counts()
mask = ~np.isin(y_encoded, class_counts[class_counts < 2].index)
X, y_encoded = X[mask], y_encoded[mask]

# Split dataset and train Random Forest classifier
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
print("Random Forest Classifier Accuracy:", accuracy)
print("Random Forest Classifier Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Visualize word frequency
def get_word_frequency(classification):
    text = " ".join(data[data['classification'] == classification]['processed_text'])
    return Counter(text.split())

for classification in pd.unique(data['classification']):
    word_freq = get_word_frequency(classification)
    common_words = word_freq.most_common(10)
    words, counts = zip(*common_words)
    plt.figure(figsize=(10, 5))
    plt.bar(words, counts)
    plt.title(f"Top 10 Words in {classification}")
    plt.show()

# Chat GPT image description generator

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="API KEY"
)
stream = client.chat.completions.create(
    model="gpt-3.5-turbo-0125",
    messages=[{"role": "user", "content": "Say this is a test"}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="API KEY"
)
response = client.chat.completions.with_raw_response.create(
    messages=[{
        "role": "user",
        "content": "Say this is a test",
    }],
    model="gpt-3.5-turbo",
)
print(response.headers.get('X-My-Header'))

completion = response.parse()  # get the object that `chat.completions.create()` would have returned
print(completion)

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="API"
)
response = client.images.generate(
  model="dall-e-3",
  prompt="a white siamese cat",
  size="1024x1024",
  quality="standard",
  n=1,
)

image_url = response.data[0].url
print(image_url)  # Print the URL for debugging purposes

In [ ]:
from IPython.display import Image, display

# Display the generated image using the image URL
display(Image(url=image_url))

These open AI calls below don't work. Please model open AI calls like those above.

In [ ]:

client = openai.OpenAI()
#base model from the website. Website is in bookmarks
#play with chat gpt and this format with data to become more comfortable
response = client.chat.completions.create(
  model="gpt-3.5-turbo-0125",
  response_format={ "type": "json_object" },
  messages=[
    {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
    {"role": "user", "content": "Who won the world series in 2020?"}
  ]
)
print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
)

print(completion.choices[0].message)


**bold text**

In [ ]:


def ask_gpt():

    model_choice = "gpt-3.5-turbo"  #@param ['gpt-3.5-turbo-16k', 'gpt-3.5-turbo', 'gpt-4']
    insert_prompt = "write me a short poem"  #@param {type: "string"}
    try:
        # Make an API call to OpenAI
        response = client.chat.completions.create(model=model_choice,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": insert_prompt}
        ])
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

print(ask_gpt())

In [ ]:


# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Initialize OpenAI API


prompt = "What are the effects of climate change?"  #@param {type: "string"}
model_choice = "gpt-3.5-turbo"  #@param ['gpt-3.5-turbo-16k', 'gpt-3.5-turbo', 'gpt-4']

def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(model=model,
        messages=[
            {"role": "system", "content": "You are a hardworking assistant."},
            {"role": "user", "content": prompt}
        ])
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Process the generated text with spaCy
doc = nlp(generated_text)

# Find relevant sentences
relevant_word = "effects"  #@param {type: "string"}
relevant_sentences = [sent.text for sent in doc.sents if relevant_word.lower() in sent.text.lower()]

print("Relevant Sentences:")
for i, sent in enumerate(relevant_sentences, 1):
    print(f"{i}. {sent}")

In [ ]:

# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Initialize OpenAI API
client = OpenAI(
    api_key="api key"

prompt = """Can you summarize the main points of the short story A Tale of the Ragged Mountains by John smith?
Making sentences to generate images for each page of the story?"""

model_choice = "gpt-3.5-turbo"  #@param ['gpt-3.5-turbo-16k', 'gpt-3.5-turbo', 'gpt-4']

def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(model=model,
        messages=[
            {"role": "system", "content": "You are a hardworking assistant."},
            {"role": "user", "content": prompt}
        ])
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Process the generated text with spaCy
doc = nlp(generated_text)

# Find relevant sentences
relevant_sentences = [sent.text for sent in doc.sents]

print("Relevant Sentences:")
for i, sent in enumerate(relevant_sentences, 1):
    print(f"{i}. {sent}")

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")
# Prompt to generate the summary text
prompt = """Can you summarize the main points of the short story A Tale of the Ragged Mountains by John Smith?
Make sentences to generate images for each page of the story."""

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Process the generated text with spaCy
doc = nlp(generated_text)

# Find and process relevant sentences
relevant_sentences = []
for sent in doc.sents:
    sentence = sent.text.replace("Page #:", "").strip()  # Remove "Page #:" and trim spaces
    tokens = [token.text for token in nlp(sentence)]  # Tokenize the sentence
    relevant_sentences.append(tokens)

print("Tokenized Sentences:")
for i, tokens in enumerate(relevant_sentences, 1):
    print(f"{i}. {tokens}")

# Loop through each tokenized sentence and generate an image
for i, tokens in enumerate(relevant_sentences, 1):
    image_prompt = " ".join(tokens)  # Convert the tokens back to a string for the image prompt
    print(f"Generating image for sentence {i}: {image_prompt}")

    # Generate the image using OpenAI DALL-E API
    try:
        response = client.images.generate(
            model="dall-e-3",
            prompt=image_prompt,
            size="1024x1024",
            quality="standard",
            n=1,
        )
        image_url = response.data[0].url
        print(image_url)  # Print the URL for debugging purposes

        # Display the generated image
        display(Image(url=image_url))
    except Exception as e:
        print(f"An error occurred while generating image for sentence {i}: {e}")

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display
import time

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")
# Author and title of the book
author = "John Smith"
title = "A Tale of the Ragged Mountains"

# Prompt to generate the summary, character list, tone, and genre
prompt = f"""Can you summarize the plot of the book titled '{title}' by {author}?
Please include the following:
1. A summary of the plot.
2. A list of all the main characters with descriptions.
3. The tone and genre of the book.
4. Ensure that each image description includes relevant details such as the tone, genre, and character appearances for the most accurate imagery.
5. and then utalize these image descriptions seperating the story to pages and make a image for each page with the given data to tell the story """

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Process the generated text with spaCy
doc = nlp(generated_text)

# Find and process relevant sentences
relevant_sentences = []
for sent in doc.sents:
    sentence = sent.text.strip()  # Trim spaces
    tokens = [token.text for token in nlp(sentence)]  # Tokenize the sentence
    relevant_sentences.append(tokens)


# Function to generate an image with error handling and rate limit management
def generate_image_with_retry(prompt, retry_limit=3):
    for attempt in range(retry_limit):
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=prompt,
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(10)
            else:
                print(f"An error occurred: {e}")
                break
    return None

# Loop through each tokenized sentence and generate an image
for i, tokens in enumerate(relevant_sentences, 1):
    image_prompt = " ".join(tokens)  # Convert the tokens back to a string for the image prompt
    print(f"Generating image for sentence {i}: {image_prompt}")

    # Generate the image with retry mechanism
    image_url = generate_image_with_retry(image_prompt)

    if image_url:
        print(image_url)  # Print the URL for debugging purposes
        # Display the generated image
        display(Image(url=image_url))
    else:
        print(f"Failed to generate image for sentence {i} after retries.")

    # Wait to ensure we don't exceed the rate limit
    time.sleep(12)  # 12 seconds between requests to stay under the 5 requests/minute limit

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display
import time

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")
# Author and title of the book
author = "edgar allan poe"
title = "A Tale of the Ragged Mountains"

# Prompt to generate the summary, character list, tone, genre, and page-by-page imagery descriptions
prompt = f"""Can you summarize the plot of the book titled '{title}' by {author}?
Please include the following:
1. A summary of the plot.
2. A list of all the main characters with descriptions.
3. The tone and genre of the book.
4. Imagery descriptions for each page of the book, ensuring they include relevant details such as the tone, genre, and character appearances for the most accurate imagery."""

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Assume the generated text is structured as needed, split into relevant sections
# Let's manually simulate splitting the text into summary, characters, and page-specific imagery descriptions

# Example summary and character list extraction (you'll have to adapt this based on actual output structure)
summary_start = generated_text.find("Summary:") + len("Summary:")
characters_start = generated_text.find("Characters:")
imagery_start = generated_text.find("Imagery Descriptions:")

summary = generated_text[summary_start:characters_start].strip()
characters = generated_text[characters_start:imagery_start].strip()
imagery_descriptions = generated_text[imagery_start + len("Imagery Descriptions:"):].strip()

# Now, process the imagery descriptions into separate pages
doc = nlp(imagery_descriptions)
pages = [sent.text.strip() for sent in doc.sents]

print("Summary:")
print(summary)
print("\nCharacters:")
print(characters)
print("\nPage Descriptions:")
for i, page in enumerate(pages, 1):
    print(f"Page {i}: {page}")

# Function to generate an image with error handling and rate limit management
def generate_image_with_retry(prompt, retry_limit=3):
    for attempt in range(retry_limit):
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=prompt,
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(10)
            else:
                print(f"An error occurred: {e}")
                break
    return None

# Loop through each page description and generate an image
for i, page_description in enumerate(pages, 1):
    image_prompt = f"{summary}\n{characters}\nPage {i} description: {page_description}"
    print(f"Generating image for Page {i} with prompt:\n{image_prompt}")


    # Generate the image with retry mechanism
    image_url = generate_image_with_retry(image_prompt)

    if image_url:
        print(image_url)  # Print the URL for debugging purposes
        # Display the generated image
        display(Image(url=image_url))
    else:
        print(f"Failed to generate image for Page {i} after retries.")

    # Wait to ensure we don't exceed the rate limit
    time.sleep(12)  # 12 seconds between requests to stay under the 5 requests/minute limit

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display
import time

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")
# Author and title of the book
author = "Edgar Allan Poe"
title = "A Tale of the Ragged Mountains"

# Prompt to generate the summary, character list, tone, genre, and page-by-page imagery descriptions
prompt = f"""Can you summarize the plot of the book titled '{title}' by {author}?
Please include the following:
1. A summary of the plot.
2. A list of all the main characters with descriptions.
3. The tone and genre of the book.
4. Imagery descriptions for each page of the book, ensuring they include relevant details such as the tone, genre, and character appearances for the most accurate imagery.
5. Ensure the imagery descriptions are suitable for generating images with no text or words included in the visuals."""

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Extract summary, characters, and imagery descriptions
summary_start = generated_text.find("Summary:") + len("Summary:")
characters_start = generated_text.find("Characters:")
imagery_start = generated_text.find("Imagery Descriptions:")

summary = generated_text[summary_start:characters_start].strip()
characters = generated_text[characters_start:imagery_start].strip()
imagery_descriptions = generated_text[imagery_start + len("Imagery Descriptions:"):].strip()

# Process the imagery descriptions into separate pages
doc = nlp(imagery_descriptions)
pages = [sent.text.strip() for sent in doc.sents]

print("Summary:")
print(summary)
print("\nCharacters:")
print(characters)
print("\nPage Descriptions (will only display the current page during generation):")

# Function to generate an image with error handling and rate limit management
def generate_image_with_retry(prompt, retry_limit=3):
    for attempt in range(retry_limit):
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=prompt,
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(10)
            else:
                print(f"An error occurred: {e}")
                break
    return None

# Loop through each page description and generate an image
for i, page_description in enumerate(pages, 1):
    image_prompt = f"{summary}\n{characters}\nPage {i} description: {page_description}\nNote: No text or words should be present in the image."
    print(f"Generating image for Page {i} with prompt:\n{page_description}")

    # Generate the image with retry mechanism
    image_url = generate_image_with_retry(image_prompt)

    if image_url:
        # Print the URL for debugging purposes
        print(image_url)
        # Display the generated image
        display(Image(url=image_url))
        # Display the text of the page
        print(f"Page {i} Text: {page_description}")
    else:
        print(f"Failed to generate image for Page {i} after retries.")

    # Wait to ensure we don't exceed the rate limit
    time.sleep(12)  # 12 seconds between requests to stay under the 5 requests/minute limit

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display
import time

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")

# Author and title of the book
author = "Edgar Allan Poe"
title = "A Tale of the Ragged Mountains"

# Prompt to generate the summary, character list, tone, genre, and page-by-page imagery descriptions
prompt = f"""Can you summarize the plot of the book titled '{title}' by {author}?
Please include the following:
1. A summary of the plot.
2. A list of all the main characters with descriptions.
3. The tone and genre of the book.
4. Imagery descriptions for each page of the book, ensuring they include relevant details such as the tone, genre, and character appearances for the most accurate imagery.
5. Ensure the imagery descriptions are suitable for generating images with no text or words included in the visuals."""

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Extract summary, characters, and imagery descriptions
summary_start = generated_text.find("Summary:") + len("Summary:")
characters_start = generated_text.find("Characters:")
imagery_start = generated_text.find("Imagery Descriptions:")

summary = generated_text[summary_start:characters_start].strip()
characters = generated_text[characters_start:imagery_start].strip()
imagery_descriptions = generated_text[imagery_start + len("Imagery Descriptions:"):].strip()

# Process the imagery descriptions into separate pages
doc = nlp(imagery_descriptions)
pages = [sent.text.strip() for sent in doc.sents]

print("Summary:")
print(summary)
print("\nCharacters:")
print(characters)
print("\nPage Descriptions (will only display the current page during generation):")

# Function to generate an image with error handling and rate limit management
def generate_image_with_retry(prompt, retry_limit=3):
    for attempt in range(retry_limit):
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=prompt,
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(10)
            else:
                print(f"An error occurred: {e}")
                break
    return None

# Loop through each page description and generate an image
for i, page_description in enumerate(pages, 1):
    image_prompt = f"{summary}\n{characters}\nPage {i} description: {page_description}\nNote: No text or words should be present in the image."
    print(f"Generating image for Page {i} with prompt:\n{page_description}")

    # Generate the image with retry mechanism
    image_url = generate_image_with_retry(image_prompt)

    if image_url:
        # Print the URL for debugging purposes
        print(image_url)
        # Display the generated image
        display(Image(url=image_url))
        # Display the text of the page
        print(f"Page {i} Text: {page_description}")
    else:
        print(f"Failed to generate image for Page {i} after retries.")

    # Wait to ensure we don't exceed the rate limit
    time.sleep(12)  # 12 seconds between requests to stay under the 5 requests/minute limit

In [ ]:
import spacy
from openai import OpenAI
from IPython.display import Image, display
import time

# Initialize spaCy and OpenAI API
nlp = spacy.load("en_core_web_sm")
client = OpenAI(api_key="API KEY")

# Author and title of the book
author = "Edgar Allen Poe"
title = "The man of the Crowd"

# Prompt to generate the summary, character list, tone, genre, and page-by-page imagery descriptions
prompt = f"""Can you summarize the plot of the book titled '{title}' by {author}?
Please include the following:
1. A summary of the plot.
2. A list of all the main characters with descriptions.
3. The tone and genre of the book.
4. Imagery descriptions for each page of the book, ensuring they include relevant details such as the tone, genre, and character appearances for the most accurate imagery.
5. Ensure the imagery descriptions are suitable for generating images with no text or words included in the visuals.
6. Do page numbers
7.Create a setting before each image, and make sure the imagery is not completely different between images"""

# Model choice
model_choice = "gpt-3.5-turbo"

# Function to generate text using OpenAI API
def generate_text(prompt, model=model_choice):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a hardworking assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"An error occurred: {e}"

# Generate the text
generated_text = generate_text(prompt)
print("Generated Text:")
print(generated_text)

# Extract summary, characters, and imagery descriptions
summary_start = generated_text.find("Summary:") + len("Summary:")
characters_start = generated_text.find("Characters:")
imagery_start = generated_text.find("Imagery Descriptions:")

summary = generated_text[summary_start:characters_start].strip()
characters = generated_text[characters_start:imagery_start].strip()
imagery_descriptions = generated_text[imagery_start + len("Imagery Descriptions:"):].strip()

# Process the imagery descriptions into separate pages
doc = nlp(imagery_descriptions)
pages = [sent.text.strip() for sent in doc.sents]

print("Summary:")
print(summary)
print("\nCharacters:")
print(characters)
print("\nPage Descriptions (will only display the current page during generation):")

# Function to generate an image with error handling and rate limit management
def generate_image_with_retry(prompt, retry_limit=3):
    for attempt in range(retry_limit):
        try:
            response = client.images.generate(
                model="dall-e-3",
                prompt=prompt,
                size="1024x1024",
                quality="standard",
                n=1,
            )
            return response.data[0].url
        except Exception as e:
            error_message = str(e)
            if "rate_limit_exceeded" in error_message:
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(10)
            else:
                print(f"An error occurred: {e}")
                break
    return None

# Loop through each page description and generate an image
for i, page_description in enumerate(pages, 1):
    image_prompt = f"{summary}\n{characters}\nPage {i} description: {page_description}\nNote: No text or words should be present in the image."
    print(f"Generating image for Page {i} with prompt:\n{page_description}")

    # Generate the image with retry mechanism
    image_url = generate_image_with_retry(image_prompt)

    if image_url:
        # Print the URL for debugging purposes
        print(image_url)
        # Display the generated image
        display(Image(url=image_url))
        # Display the text of the page
        print(f"Page {i} Text: {page_description}")
    else:
        print(f"Failed to generate image for Page {i} after retries.")

    # Wait to ensure we don't exceed the rate limit
    time.sleep(12)  # 12 seconds between requests to stay under the 5 requests/minute limit